# **Dataset Importing**

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mirichoi0218/insurance")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\mrayy\.cache\kagglehub\datasets\mirichoi0218\insurance\versions\1


In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


files = os.listdir(path)
print("Files in dataset:", files)


df = pd.read_csv(os.path.join(path, "insurance.csv"))

ModuleNotFoundError: No module named 'matplotlib.backends.registry'

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

# **Visualization**

In [ ]:
plt.hist(df["age"], color="aqua",  edgecolor="black")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.show()

In [ ]:
sk_Age = df["age"].skew()
print(f"Skewness is: {sk_Age:.5f}")

In [ ]:
plt.hist(df["sex"], color="aqua",  edgecolor="black")
plt.xlabel("Gender")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.hist(df["bmi"], color="aqua",  edgecolor="black")
plt.xlabel("BMI")
plt.ylabel("Frequency")
plt.show()

In [ ]:
sk_BMI = df["bmi"].skew()
print(f"Skewness is: {sk_BMI:.5f}")

In [ ]:
plt.hist(df["children"], color="aqua",  edgecolor="black")
plt.xlabel("Children")
plt.ylabel("Frequency")
plt.show()

In [ ]:
sk_Children = df["children"].skew()
print(f"Skewness is: {sk_Children:.5f}")

In [ ]:
plt.hist(df["smoker"], color="aqua",  edgecolor="black")
plt.xlabel("Smokers")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.hist(df["region"], color="aqua",  edgecolor="black")
plt.xlabel("Region")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.hist(df["charges"], color="aqua",  edgecolor="black")
plt.xlabel("Charges")
plt.ylabel("Frequency")
plt.show()

In [ ]:
sk_Charges = df["charges"].skew()
print(f"Skewness is: {sk_Charges:.5f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

df["children"].plot.kde(ax=axes[0], title="Children KDE")
df["bmi"].plot.kde(ax=axes[1], title="BMI KDE")
df["charges"].plot.kde(ax=axes[2], title="Charges KDE")

axes[0].set_xlabel("children")
axes[1].set_xlabel("bmi")
axes[2].set_xlabel("charges")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes  = plt.subplots(1,2, figsize=(12,5))

axes[0].boxplot(df["children"])
axes[0].set_title("Skewness in Children Column")


axes[1].boxplot(df["charges"])
axes[1].set_title("Skewness in Charges Column")

plt.show()

In [ ]:
plt.scatter(df["children"],df["charges"] ,s=50, colorizer="red")

In [ ]:
df.info()

In [ ]:
# Encode categorical features
model_df = pd.get_dummies(df, columns=["sex", "smoker", "region"], drop_first=True)

In [ ]:
correlation_matrix = model_df[["age","bmi","children","charges"]].corr()
print(correlation_matrix)
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# vertical split
X = model_df.drop("charges", axis=1)
# y = model_df["charges"]

y = np.log(df["charges"])


In [ ]:
model_df[["bmi", "children", "charges"]].plot.kde()

In [ ]:
from sklearn.preprocessing import StandardScaler,RobustScaler

scaler = RobustScaler()
scaled_data = scaler.fit_transform(model_df[["bmi", "children", "charges"]])
scaled_df = pd.DataFrame(scaled_data, columns=["bmi", "children", "charges"])


In [ ]:
scaled_df[["bmi", "children", "charges"]].plot.kde()

# **Model Training**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler0 = StandardScaler()
X_train_scaled = scaler0.fit_transform(X_train)
X_test_scaled = scaler0.transform(X_test)

In [ ]:
# Regression modeling with Linear Regression

from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred = lr.predict(X_test_scaled)

# After prediction, reverse it:
y_pred_actual = np.exp(y_pred)


In [ ]:
lr.coef_

In [ ]:
model_df.head()

# **Deature Importance**

In [ ]:
# Feature Importance

# 1. Coefficients nikalen (Absolute values lein taakay +ive/-ive dono ka asar dikhe)
importances = np.abs(lr.coef_) 
features = X.columns # Ya aapki feature list

# 2. Sorting 
indices = np.argsort(importances)

# 3. Ploting
plt.figure(figsize=(10, 6))
plt.title('Feature Importance (Based on Coefficients)')
plt.barh(range(len(indices)), importances[indices], color='orange', align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Absolute Coefficient Value')
plt.show()


In [ ]:
# Visualizations: Age, BMI, Smoker effect on charges
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sns.scatterplot(x="age", y="charges", data=df, hue="smoker", ax=axes[0])
axes[0].set_title("Age vs Charges")

sns.scatterplot(x="bmi", y="charges", data=df, hue="smoker", ax=axes[1])
axes[1].set_title("BMI vs Charges")

sns.boxplot(x="smoker", y="charges", data=df, ax=axes[2], width=0.3, color="lightgreen")
axes[2].set_title("Smoker vs Charges")

plt.tight_layout()
plt.show()

# **Metrices Calculation**

In [ ]:
# Regression task: don’t use accuracy_score on continuous targets.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score,classification_report,confusion_matrix

y_pred = lr.predict(X_test_scaled)

y_pred_actual = np.exp(y_pred)

mae = mean_absolute_error(y_test, y_pred_actual)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_actual))
r2 = r2_score(y_test, y_pred_actual)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {r2:.4f}")


# Classification-style evaluation via binarized target (high cost vs low cost)
median_charge = df['charges'].median()
y_test_class = (y_test > median_charge).astype(int)
y_pred_class = (y_pred_actual > median_charge).astype(int)

from sklearn.metrics import classification_report, confusion_matrix

print("Classification report (high-cost=1, low-cost=0):")
print(classification_report(y_test_class, y_pred_class))

cm = confusion_matrix(y_test_class, y_pred_class)
print("Confusion matrix:")
print(cm)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()